# Phase 5: Statistical Inference
Confidence intervals, non-parametric hypothesis tests, and a formal model comparison —
confirming the Phase 4 findings are statistically sound, not just visually suggestive.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from dotenv import load_dotenv
from scipy import stats
import numpy as np
import os

load_dotenv()
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

model_df = pd.read_sql("SELECT * FROM order_modeling_dataset;", engine)
model_df = model_df[~model_df["order_status"].isin(["canceled", "unavailable"])]
model_df = model_df[model_df["review_score"].notna()]
model_df = model_df[model_df["delivery_days"].notna()]
model_df = model_df[model_df["avg_distance_km"].notna()]

model_df["review_score_cat"] = pd.Categorical(model_df["review_score"], categories=[1, 2, 3, 4, 5], ordered=True)
top_categories = model_df["primary_category"].value_counts().nlargest(15).index
model_df["category_grouped"] = model_df["primary_category"].where(model_df["primary_category"].isin(top_categories), "other")
model_df["is_late_num"] = model_df["is_late"].astype(int)

for col in ["delivery_days", "avg_distance_km", "order_value", "freight_value", "item_count"]:
    model_df[col + "_z"] = (model_df[col] - model_df[col].mean()) / model_df[col].std()

print(model_df.shape)

(95349, 22)


## 1. Confidence intervals on the key coefficients
A p-value tells you IF an effect is real; a confidence interval tells you the
plausible RANGE of how big it is — more useful when writing an actual recommendation.

In [8]:
from statsmodels.miscmodels.ordinal_model import OrderedModel

formula = (
    "review_score_cat ~ delivery_days_z + is_late_num + avg_distance_km_z "
    "+ order_value_z + freight_value_z + item_count_z "
    "+ C(category_grouped) + C(customer_state)"
)
full_model = OrderedModel.from_formula(formula, data=model_df, distr="logit")
full_result = full_model.fit(method="bfgs", maxiter=1000, disp=False)

ci = full_result.conf_int(alpha=0.05)
ci.columns = ["ci_lower", "ci_upper"]
key_vars = ["delivery_days_z", "is_late_num", "avg_distance_km_z", "item_count_z"]

ci_table = pd.DataFrame({
    "coef": full_result.params[key_vars],
    "ci_lower": ci.loc[key_vars, "ci_lower"],
    "ci_upper": ci.loc[key_vars, "ci_upper"],
})
# Convert to odds ratios for readability
ci_table["odds_ratio"] = np.exp(ci_table["coef"])
ci_table["or_ci_lower"] = np.exp(ci_table["ci_lower"])
ci_table["or_ci_upper"] = np.exp(ci_table["ci_upper"])

ci_table["pct_change"] = (1 - ci_table["odds_ratio"]) * 100
ci_table["_a"] = (1 - ci_table["or_ci_lower"]) * 100
ci_table["_b"] = (1 - ci_table["or_ci_upper"]) * 100
ci_table["pct_change_low"] = ci_table[["_a", "_b"]].min(axis=1)
ci_table["pct_change_high"] = ci_table[["_a", "_b"]].max(axis=1)
ci_table = ci_table.drop(columns=["_a", "_b"])
 
ci_table[["coef", "odds_ratio", "or_ci_lower", "or_ci_upper", "pct_change", "pct_change_low", "pct_change_high"]]

,coef,odds_ratio,or_ci_lower,or_ci_upper,pct_change,pct_change_low,pct_change_high
delivery_days_z,-0.508606,0.601333,0.589152,0.613766,39.866701,38.623428,41.084789
is_late_num,-1.299755,0.272599,0.257102,0.289030,72.740131,71.097018,74.289835
avg_distance_km_z,0.034340,1.034936,1.008130,1.062456,-3.493638,-6.245606,-0.812951
item_count_z,-0.224128,0.799213,0.787164,0.811446,20.078737,18.855421,21.283611


point estimates from phase 4 weren't unlucky draws!!
Delivery Day: One Standard Deviation increase in delivery time is associated with 38.6% - 41.1% LOWER odds of a higher review score with 95% confidence.

Late Order: Being late REDUCES odds of higher review by 71.1% - 74.3%

Item Count: One Standard Deviation increase in item count REDUCES odds of higher review by 18.9% - 21.3%

Average Distance: One Standard deviation increase in distance INCREASES odds of a higher review by 0.8% - 6.2%

## 2. Non-parametric hypothesis tests
Review scores are heavily skewed (Phase 3), so a standard t-test's normality
assumption doesn't hold well. Mann-Whitney U and Kruskal-Wallis work on RANKS
instead of raw values, making no assumption about the shape of the distribution.

In [3]:
# Mann-Whitney U: is the distribution of review scores different for late vs on-time orders?
late_scores = model_df.loc[model_df["is_late"] == True, "review_score"]
ontime_scores = model_df.loc[model_df["is_late"] == False, "review_score"]

u_stat, u_pval = stats.mannwhitneyu(late_scores, ontime_scores, alternative="two-sided")
print(f"Mann-Whitney U: statistic={u_stat:.1f}, p-value={u_pval:.2e}")

Mann-Whitney U: statistic=148872387.0, p-value=0.00e+00


Difference in review score distributions between late and on time orders is statistically certain. 
Regression Quantifies Effect. 
Mann Whitney parametric test confirms its real

In [4]:
# Kruskal-Wallis: do review scores differ across delivery-speed buckets? (generalizes
# Mann-Whitney to more than 2 groups — our original naive baseline buckets from Phase 2)
def bucket(days):
    if days <= 3: return "0-3 days"
    elif days <= 7: return "4-7 days"
    elif days <= 14: return "8-14 days"
    elif days <= 21: return "15-21 days"
    else: return "22+ days"

model_df["delivery_bucket"] = model_df["delivery_days"].apply(bucket)
groups = [g["review_score"].values for _, g in model_df.groupby("delivery_bucket")]

h_stat, h_pval = stats.kruskal(*groups)
print(f"Kruskal-Wallis H: statistic={h_stat:.1f}, p-value={h_pval:.2e}")

Kruskal-Wallis H: statistic=7841.3, p-value=0.00e+00


Across all 5 delivery speed confounders, review score distributions are definitely not the same

## 3. Likelihood ratio test: do the delivery variables actually earn their place?
Compares the full model against a reduced model with delivery_days and is_late
REMOVED, testing whether they jointly improve model fit beyond the confounders alone.

In [5]:
reduced_formula = (
    "review_score_cat ~ avg_distance_km_z + order_value_z + freight_value_z + item_count_z "
    "+ C(category_grouped) + C(customer_state)"
)
reduced_model = OrderedModel.from_formula(reduced_formula, data=model_df, distr="logit")
reduced_result = reduced_model.fit(method="bfgs", maxiter=1000, disp=False)

lr_stat = 2 * (full_result.llf - reduced_result.llf)
df_diff = full_result.df_model - reduced_result.df_model
lr_pval = stats.chi2.sf(lr_stat, df_diff)

print(f"Full model log-likelihood:    {full_result.llf:.1f}")
print(f"Reduced model log-likelihood: {reduced_result.llf:.1f}")
print(f"Likelihood ratio statistic: {lr_stat:.2f} (df={df_diff})")
print(f"p-value: {lr_pval:.2e}")

Full model log-likelihood:    -104877.1
Reduced model log-likelihood: -110563.9
Likelihood ratio statistic: 11373.64 (df=2)
p-value: 0.00e+00


Removing delivery date and is_late makes the log likelihood much worse and LR statistic is giant.
Delivery date and is late are doing real work in explaining customer satisfaction